# 4054. Count Shadow Pairs I — Nhập môn Monotonic Stack (ngăn xếp đơn điệu)

Bài toán LeetCode (Medium):

```python
class Solution:
    def shadowPairs(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu của notebook

Sau notebook này, bạn sẽ hiểu:

- Cách đọc một điều kiện phức tạp và **rút gọn** nó về dạng dễ hiểu ("cái bóng" của một phần tử).
- Lời giải Brute Force `O(n^3)` đúng theo định nghĩa, rồi cải tiến `O(n^2)` bằng **dừng sớm**.
- Cấu trúc dữ liệu **stack** (ngăn xếp) trong Python: `append`, `pop`, `[-1]`.
- Kỹ thuật **Monotonic Stack**: giữ một stack luôn **không giảm**, loại bỏ các ứng viên "đã chết".
- Kỹ thuật **gom nhóm giá trị bằng nhau** + **biến tổng** để đạt `O(n)`.
- (Mở rộng) Cách thứ hai bằng **Next Smaller Element + Fenwick Tree**.
- Trace, code chuẩn LeetCode, lỗi hay mắc, biến thể, bài tập có đáp án.

# 1. Đề bài nói gì?

Cho mảng `nums` độ dài `n`. Cặp chỉ số `(i, j)` là **shadow pair** nếu thoả **cả ba**:

1. `0 <= i < j < n`
2. `nums[i] < nums[j]`
3. **Không tồn tại** `k` với `i < k < j` mà `nums[k] < nums[i] < nums[j]`.

Đếm số shadow pair.

### Ví dụ

| nums | Kết quả | Các cặp |
|---|---:|---|
| `[3, 1, 4, 1, 5]` | 3 | `(1,2)`, `(1,4)`, `(3,4)` |
| `[6, 7, 6, 6, 7]` | 4 | `(0,1)`, `(0,4)`, `(2,4)`, `(3,4)` |
| `[1, 2, 3, 4]` | 6 | mọi cặp `i < j` |

### Ràng buộc

- `3 <= n <= 10^5`
- `1 <= nums[i] <= 10^9`

# 2. Rút gọn điều kiện thứ 3

Điều kiện 3 nói: không có `k` ở giữa mà `nums[k] < nums[i] < nums[j]`.

Nhưng điều kiện 2 **đã** đảm bảo `nums[i] < nums[j]`. Vậy phần `nums[i] < nums[j]` trong điều kiện 3 luôn đúng, và điều kiện 3 chỉ còn là:

> **Không có phần tử nào ở giữa `i` và `j` nhỏ hơn hẳn `nums[i]`.**

Tức là: mọi phần tử nằm giữa đều `>= nums[i]`.

## Hình ảnh "cái bóng"

Hãy tưởng tượng phần tử `nums[i]` **đổ bóng** sang bên phải. Bóng kéo dài cho đến khi gặp phần tử **nhỏ hơn hẳn** `nums[i]` thì bị chặn lại.

> `(i, j)` là shadow pair ⇔ `j` nằm **trong bóng** của `i` **và** `nums[j] > nums[i]`.

Ví dụ `nums = [3, 1, 4, 1, 5]`:

```text
chỉ số:  0  1  2  3  4
nums:    3  1  4  1  5

bóng của i=0 (giá trị 3): bị chặn ngay tại k=1 (1 < 3)  -> không có j nào
bóng của i=1 (giá trị 1): kéo tới hết mảng (không có ai < 1)
         j=2 (4 > 1) ✔   j=3 (1, không lớn hơn) ✘   j=4 (5 > 1) ✔
bóng của i=2 (giá trị 4): bị chặn tại k=3 (1 < 4)   -> không có j nào
bóng của i=3 (giá trị 1): j=4 (5 > 1) ✔
```

Tổng: 3 cặp ✔. Chú ý: phần tử **bằng** `nums[i]` **không** chặn bóng (ví dụ 2), nhưng cũng **không** tạo cặp.

# 3. Cách 1 — Brute Force đúng theo định nghĩa: `O(n^3)`

Dịch thẳng ba điều kiện ra code. Hàm `any(...)` trả về `True` nếu **có ít nhất một** phần tử thoả điều kiện.

In [1]:
print(any(x < 0 for x in [3, 5, -1]))   # True: có -1 < 0
print(any(x < 0 for x in [3, 5, 1]))    # False
print(any(x < 0 for x in []))           # False: không có phần tử nào

True
False
False


In [2]:
def shadow_pairs_cubic(nums):
    n = len(nums)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] < nums[j]:
                blocked = any(nums[k] < nums[i] for k in range(i + 1, j))
                if not blocked:
                    count += 1
    return count


print(shadow_pairs_cubic([3, 1, 4, 1, 5]))   # 3
print(shadow_pairs_cubic([6, 7, 6, 6, 7]))   # 4
print(shadow_pairs_cubic([1, 2, 3, 4]))      # 6

3
4
6


Ba vòng lặp lồng nhau → `O(n^3)`. Với `n = 10^5`, không thể chạy.

# 4. Cách 2 — Dừng sớm khi bóng bị chặn: `O(n^2)`

Nhận xét: với mỗi `i`, khi đi `j` sang phải, nếu gặp phần tử **nhỏ hơn** `nums[i]` thì **mọi `j` phía sau đều bị chặn** → `break` ngay.

Trong lúc đi, gặp phần tử **lớn hơn** thì đếm. Không cần vòng lặp `k` nữa.

In [3]:
def shadow_pairs_quadratic(nums):
    n = len(nums)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if nums[j] < nums[i]:      # bóng bị chặn
                break
            if nums[j] > nums[i]:      # trong bóng và lớn hơn
                count += 1
            # nums[j] == nums[i]: không chặn, không đếm
    return count


print(shadow_pairs_quadratic([3, 1, 4, 1, 5]))   # 3
print(shadow_pairs_quadratic([6, 7, 6, 6, 7]))   # 4
print(shadow_pairs_quadratic([1, 2, 3, 4]))      # 6

3
4
6


Vẫn `O(n^2)` trong trường hợp xấu nhất (mảng tăng dần: bóng của ai cũng kéo tới cuối). Với `n = 10^5` là `5·10^9` bước — quá chậm.

Đáp án cũng có thể rất lớn (mảng tăng dần cho `n(n-1)/2 ≈ 5·10^9` cặp), nên ta phải **đếm theo nhóm** chứ không đếm từng cặp.

# 5. Đổi góc nhìn: cố định `j`, hỏi "những `i` nào còn sống?"

Thay vì hỏi "bóng của `i` chứa những `j` nào", ta đi `j` từ trái sang phải và hỏi:

> Tại thời điểm `j`, những `i < j` nào có bóng **vẫn còn kéo tới `j`**?

Gọi những `i` đó là **ứng viên còn sống**. Khi một phần tử mới `x = nums[j]` xuất hiện:

1. Mọi ứng viên có giá trị **lớn hơn `x`** bị `x` **chặn bóng vĩnh viễn** → loại bỏ (chết).
2. Trong các ứng viên còn lại (giá trị `<= x`), những ai có giá trị **nhỏ hơn hẳn `x`** tạo shadow pair với `j`.
3. Bản thân `j` trở thành một ứng viên mới.

### Một tính chất quan trọng

Sau bước 1, mọi ứng viên còn sống đều `<= x`, rồi ta thêm `x` vào cuối. Vì vậy, nếu ta xếp các ứng viên theo thứ tự xuất hiện, **giá trị của chúng không giảm**:

```text
ứng viên còn sống (theo thứ tự trái → phải):  v1 <= v2 <= v3 <= ... <= vk
```

Nghĩa là các ứng viên **lớn hơn `x`** luôn nằm **ở cuối** dãy. Muốn loại bỏ chúng, chỉ cần **xoá từ cuối** cho đến khi gặp phần tử `<= x`.

"Thêm vào cuối, xoá từ cuối" — đó chính là **stack**!

# 6. Nền tảng Python — Stack (ngăn xếp)

Stack giống **chồng đĩa**: đĩa đặt sau cùng sẽ được lấy ra đầu tiên (LIFO — Last In, First Out).

Trong Python, `list` dùng làm stack rất tốt:

| Thao tác | Code | Độ phức tạp |
|---|---|---|
| đặt lên đỉnh | `stack.append(x)` | `O(1)` |
| lấy ra từ đỉnh | `stack.pop()` | `O(1)` |
| xem đỉnh | `stack[-1]` | `O(1)` |
| kiểm tra rỗng | `if stack:` / `if not stack:` | `O(1)` |

In [4]:
stack = []
stack.append(3)
stack.append(5)
stack.append(8)
print("stack:", stack, "| đỉnh:", stack[-1])
print("pop ra:", stack.pop())
print("stack:", stack)
print("stack có rỗng không?", not stack)

stack: [3, 5, 8] | đỉnh: 8
pop ra: 8
stack: [3, 5]
stack có rỗng không? False


## "Monotonic stack" — stack đơn điệu

Monotonic stack là stack mà ta **giữ cho các phần tử luôn có thứ tự** (ở đây: không giảm từ đáy lên đỉnh). Trước khi `append(x)`, ta `pop` hết các phần tử vi phạm thứ tự (lớn hơn `x`).

In [5]:
def build_non_decreasing_stack(nums):
    stack = []
    for x in nums:
        while stack and stack[-1] > x:     # đỉnh lớn hơn x -> bị x chặn -> loại
            stack.pop()
        stack.append(x)
        print(f"x = {x}: stack = {stack}")

build_non_decreasing_stack([3, 1, 4, 1, 5])

x = 3: stack = [3]
x = 1: stack = [1]
x = 4: stack = [1, 4]
x = 1: stack = [1, 1]
x = 5: stack = [1, 1, 5]


# 7. Cách 3 — Monotonic Stack, đếm bằng cách duyệt stack

Áp dụng mục 5:

```text
với mỗi x = nums[j]:
    pop mọi phần tử > x                         (ứng viên chết)
    đếm số phần tử trong stack có giá trị < x   (tạo cặp với j)
    push x
```

Ta đếm bằng cách duyệt cả stack — đơn giản, dễ kiểm chứng, nhưng chưa tối ưu.

In [6]:
def shadow_pairs_stack_scan(nums):
    stack = []
    count = 0
    for x in nums:
        while stack and stack[-1] > x:
            stack.pop()
        count += sum(1 for v in stack if v < x)     # duyệt stack để đếm
        stack.append(x)
    return count


print(shadow_pairs_stack_scan([3, 1, 4, 1, 5]))   # 3
print(shadow_pairs_stack_scan([6, 7, 6, 6, 7]))   # 4
print(shadow_pairs_stack_scan([1, 2, 3, 4]))      # 6

3
4
6


### Vẫn chậm ở đâu?

Phần `pop` tổng cộng chỉ `O(n)` (mỗi phần tử vào stack 1 lần, ra nhiều nhất 1 lần). Nhưng phần `sum(...)` duyệt cả stack mỗi bước: với mảng tăng dần, stack dài dần `1, 2, 3, ...` → tổng `O(n^2)`.

Cần đếm "số phần tử `< x` trong stack" **trong `O(1)`**.

# 8. Bước đột phá — Stack không giảm nên các phần tử `== x` nằm ở đỉnh

Sau khi pop hết phần tử `> x`, stack chỉ còn giá trị `<= x`, và vì không giảm, các phần tử **bằng `x`** (nếu có) đều nằm **sát đỉnh**:

```text
stack (đáy → đỉnh):   1  2  2  5  6  6  6      x = 6
                      \_______/  \_____/
                       < x (4 cái)  == x (3 cái)
```

Vậy:

```text
số phần tử < x  =  (tổng số phần tử trong stack)  -  (số phần tử == x ở đỉnh)
```

Để có cả hai con số trong `O(1)`:

1. **Gom nhóm**: mỗi ô stack lưu cặp `[giá_trị, số_lượng]` thay vì từng phần tử riêng lẻ.
2. Giữ biến `total` = tổng số phần tử (tổng các `số_lượng`) trong stack.

In [7]:
def shadow_pairs_grouped_stack(nums):
    stack = []          # mỗi ô: [value, count], value tăng dần từ đáy lên đỉnh
    total = 0           # tổng số ứng viên còn sống
    answer = 0
    for x in nums:
        # 1) loại các nhóm lớn hơn x
        while stack and stack[-1][0] > x:
            total -= stack.pop()[1]
        # 2) đếm ứng viên < x, rồi 3) thêm x vào stack
        if stack and stack[-1][0] == x:
            answer += total - stack[-1][1]     # trừ nhóm bằng x ở đỉnh
            stack[-1][1] += 1                  # gộp x vào nhóm đó
        else:
            answer += total                    # mọi ứng viên đều < x
            stack.append([x, 1])
        total += 1
    return answer


print(shadow_pairs_grouped_stack([3, 1, 4, 1, 5]))   # 3
print(shadow_pairs_grouped_stack([6, 7, 6, 6, 7]))   # 4
print(shadow_pairs_grouped_stack([1, 2, 3, 4]))      # 6

3
4
6


`stack.pop()[1]`: `pop()` trả về ô vừa lấy ra (một list `[value, count]`), và `[1]` lấy luôn `count` của nó. Một dòng làm hai việc: lấy ra và trừ số lượng khỏi `total`.

# 9. Mô phỏng từng bước

Ví dụ `nums = [6, 7, 6, 6, 7]` (có nhiều giá trị bằng nhau — ví dụ "khó"):

| j | x | Pop (nhóm > x) | stack sau pop | total | nhóm đỉnh == x? | cộng | answer | stack sau push | total sau |
|---|---|---|---|---:|---|---:|---:|---|---:|
| 0 | 6 | — | `[]` | 0 | không | 0 | 0 | `[[6,1]]` | 1 |
| 1 | 7 | — | `[[6,1]]` | 1 | không | 1 | 1 | `[[6,1],[7,1]]` | 2 |
| 2 | 6 | `[7,1]` | `[[6,1]]` | 1 | có (1 cái) | 1 − 1 = 0 | 1 | `[[6,2]]` | 2 |
| 3 | 6 | — | `[[6,2]]` | 2 | có (2 cái) | 2 − 2 = 0 | 1 | `[[6,3]]` | 3 |
| 4 | 7 | — | `[[6,3]]` | 3 | không | 3 | **4** | `[[6,3],[7,1]]` | 4 |

Ở bước `j = 4`: ba số `6` (tại `i = 0, 2, 3`) đều còn sống và nhỏ hơn 7 → 3 cặp `(0,4), (2,4), (3,4)`. Chú ý số `7` ở `i = 1` đã chết tại `j = 2` (bị số 6 chặn).

In [8]:
def trace_shadow(nums):
    stack, total, answer = [], 0, 0
    for j, x in enumerate(nums):
        popped = []
        while stack and stack[-1][0] > x:
            group = stack.pop()
            total -= group[1]
            popped.append(group)
        if stack and stack[-1][0] == x:
            add = total - stack[-1][1]
            stack[-1][1] += 1
        else:
            add = total
            stack.append([x, 1])
        answer += add
        total += 1
        print(f"j={j}, x={x}: pop {popped if popped else '-'} | cộng {add} | answer={answer} | stack={stack} | total={total}")
    return answer

trace_shadow([6, 7, 6, 6, 7])
print()
trace_shadow([3, 1, 4, 1, 5])

j=0, x=6: pop - | cộng 0 | answer=0 | stack=[[6, 1]] | total=1
j=1, x=7: pop - | cộng 1 | answer=1 | stack=[[6, 1], [7, 1]] | total=2
j=2, x=6: pop [[7, 1]] | cộng 0 | answer=1 | stack=[[6, 2]] | total=2
j=3, x=6: pop - | cộng 0 | answer=1 | stack=[[6, 3]] | total=3
j=4, x=7: pop - | cộng 3 | answer=4 | stack=[[6, 3], [7, 1]] | total=4

j=0, x=3: pop - | cộng 0 | answer=0 | stack=[[3, 1]] | total=1
j=1, x=1: pop [[3, 1]] | cộng 0 | answer=0 | stack=[[1, 1]] | total=1
j=2, x=4: pop - | cộng 1 | answer=1 | stack=[[1, 1], [4, 1]] | total=2
j=3, x=1: pop [[4, 1]] | cộng 0 | answer=1 | stack=[[1, 2]] | total=2
j=4, x=5: pop - | cộng 2 | answer=3 | stack=[[1, 2], [5, 1]] | total=3


3

# 10. Lời giải LeetCode chuẩn

In [9]:
class Solution:
    def shadowPairs(self, nums: list[int]) -> int:
        stack = []      # [value, count], value không giảm từ đáy lên đỉnh
        total = 0       # số ứng viên còn sống
        answer = 0
        for x in nums:
            while stack and stack[-1][0] > x:
                total -= stack.pop()[1]
            if stack and stack[-1][0] == x:
                answer += total - stack[-1][1]
                stack[-1][1] += 1
            else:
                answer += total
                stack.append([x, 1])
            total += 1
        return answer

In [10]:
sol = Solution()

tests = [
    ([3, 1, 4, 1, 5], 3),
    ([6, 7, 6, 6, 7], 4),
    ([1, 2, 3, 4], 6),
    ([4, 3, 2, 1], 0),            # giảm dần: mọi bóng bị chặn ngay
    ([5, 5, 5], 0),               # bằng nhau: không có cặp
    ([1, 3, 2], 2),               # (0,1), (0,2); (1,2) sai vì 2 < 3
    ([2, 5, 1, 3, 4], 4),
    ([1, 10**9, 1], 1),
]
for nums, expected in tests:
    result = sol.shadowPairs(nums)
    print(f"{str(nums):22} -> {result} | expected {expected} | {'PASS' if result == expected else 'FAIL'}")
    assert result == expected

[3, 1, 4, 1, 5]        -> 3 | expected 3 | PASS
[6, 7, 6, 6, 7]        -> 4 | expected 4 | PASS
[1, 2, 3, 4]           -> 6 | expected 6 | PASS
[4, 3, 2, 1]           -> 0 | expected 0 | PASS
[5, 5, 5]              -> 0 | expected 0 | PASS
[1, 3, 2]              -> 2 | expected 2 | PASS
[2, 5, 1, 3, 4]        -> 4 | expected 4 | PASS
[1, 1000000000, 1]     -> 1 | expected 1 | PASS


In [11]:
import random, time

random.seed(54)
for _ in range(2000):
    nums = [random.randint(1, 5) for _ in range(random.randint(3, 10))]
    b = shadow_pairs_cubic(nums)
    assert shadow_pairs_quadratic(nums) == b
    assert shadow_pairs_stack_scan(nums) == b
    assert sol.shadowPairs(nums) == b
print("4 cách cho cùng kết quả trên 2000 mảng ngẫu nhiên ✔")

inc = list(range(1, 10**5 + 1))          # trường hợp xấu nhất cho cách 2, 3
t0 = time.time()
print("Mảng tăng dần n=1e5:", sol.shadowPairs(inc), "cặp", f"({time.time() - t0:.3f} giây)")

4 cách cho cùng kết quả trên 2000 mảng ngẫu nhiên ✔
Mảng tăng dần n=1e5: 4999950000 cặp (0.022 giây)


# 11. Vì sao thuật toán đúng?

**Bất biến của vòng lặp:** ngay trước khi xử lý `j`, stack chứa (dưới dạng nhóm) **đúng** các chỉ số `i < j` mà mọi phần tử giữa `i` và `j` đều `>= nums[i]` — tức các `i` có bóng còn kéo tới `j`. Và giá trị trong stack không giảm.

- **Pop đúng:** phần tử `x = nums[j]` chặn bóng của mọi `i` có `nums[i] > x`. Những `i` này sẽ không bao giờ tạo cặp với `j' > j` nữa (vì `j` nằm giữa). Do stack không giảm, chúng nằm liền ở đỉnh.
- **Đếm đúng:** các `i` còn lại có `nums[i] <= x` và bóng kéo tới `j`. `(i, j)` là shadow pair ⇔ thêm `nums[i] < x` ⇔ không thuộc nhóm `== x` ở đỉnh.
- **Push đúng:** `j` có bóng (rỗng) kéo tới `j + 1`, nên là ứng viên cho bước sau. Thêm `x` vào đỉnh giữ stack không giảm.

Mỗi shadow pair `(i, j)` được đếm đúng một lần — tại bước `j`.

# 12. Độ phức tạp

Gọi `n = len(nums)`.

| Cách | Thời gian | Bộ nhớ phụ |
|---|---|---|
| 1. Theo định nghĩa | `O(n^3)` | `O(1)` |
| 2. Dừng sớm | `O(n^2)` | `O(1)` |
| 3. Stack + duyệt stack | `O(n^2)` worst | `O(n)` |
| 4. Stack gom nhóm | `O(n)` | `O(n)` |

Vì sao cách 4 là `O(n)` dù có vòng `while` bên trong `for`? **Phân tích khấu hao** (amortized): mỗi nhóm được `append` tối đa một lần cho mỗi phần tử và `pop` tối đa một lần. Tổng số lần `pop` qua cả thuật toán `<= n`. Vậy tổng công việc `O(n)`.

# 13. Cách 5 (mở rộng) — Next Smaller Element + Fenwick Tree

Phần này dành cho bạn muốn thấy **một lối tư duy khác**. Có thể bỏ qua ở lần đọc đầu.

## Ý tưởng

Với mỗi `i`, gọi `r[i]` là chỉ số đầu tiên bên phải có `nums[r[i]] < nums[i]` (nếu không có thì `r[i] = n`). Đó chính là **chỗ bóng bị chặn**.

```text
đóng góp của i = số j trong khoảng (i, r[i]) có nums[j] > nums[i]
```

**Bước 1 — tính `r[i]`:** cũng bằng monotonic stack (bài kinh điển "Next Smaller Element").

**Bước 2 — đếm "số phần tử lớn hơn `nums[i]` trong một đoạn chỉ số":** xử lý các giá trị theo thứ tự **giảm dần**. Khi xét giá trị `v`, ta đã "đánh dấu" vị trí của mọi phần tử `> v`. Số phần tử được đánh dấu trong đoạn `(i, r[i])` chính là đóng góp. Cần một cấu trúc **đánh dấu một vị trí** và **đếm số vị trí được đánh dấu trong một đoạn** nhanh → **Fenwick Tree** (Binary Indexed Tree).

## Fenwick Tree trong 30 giây

Một mảng `tree` cho phép, trong `O(log n)`:

- `add(pos)`: đánh dấu vị trí `pos`.
- `prefix(pos)`: đếm số vị trí được đánh dấu trong `1..pos`.

Mẹo cốt lõi là `pos & (-pos)` — **bit 1 thấp nhất** của `pos`, quyết định mỗi ô `tree[pos]` "phụ trách" bao nhiêu vị trí.

In [12]:
for pos in [1, 2, 3, 4, 6, 8, 12]:
    print(f"pos = {pos:2} = {pos:04b}₂ -> pos & -pos = {pos & -pos}")

pos =  1 = 0001₂ -> pos & -pos = 1
pos =  2 = 0010₂ -> pos & -pos = 2
pos =  3 = 0011₂ -> pos & -pos = 1
pos =  4 = 0100₂ -> pos & -pos = 4
pos =  6 = 0110₂ -> pos & -pos = 2
pos =  8 = 1000₂ -> pos & -pos = 8
pos = 12 = 1100₂ -> pos & -pos = 4


In [13]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)          # dùng chỉ số 1..n

    def add(self, pos, value=1):
        while pos <= self.n:
            self.tree[pos] += value
            pos += pos & (-pos)

    def prefix(self, pos):                 # tổng trên 1..pos
        s = 0
        while pos > 0:
            s += self.tree[pos]
            pos -= pos & (-pos)
        return s


fw = Fenwick(8)
for p in [2, 5, 6]:
    fw.add(p)
print("số vị trí được đánh dấu trong 1..5:", fw.prefix(5))                 # 2 (vị trí 2, 5)
print("số vị trí được đánh dấu trong 3..6:", fw.prefix(6) - fw.prefix(2))  # 2 (vị trí 5, 6)

số vị trí được đánh dấu trong 1..5: 2
số vị trí được đánh dấu trong 3..6: 2


In [14]:
def next_smaller_index(nums):
    n = len(nums)
    r = [n] * n
    stack = []                       # chứa chỉ số, giá trị không giảm
    for j, x in enumerate(nums):
        while stack and nums[stack[-1]] > x:
            r[stack.pop()] = j       # x là phần tử nhỏ hơn đầu tiên bên phải
        stack.append(j)
    return r


def shadow_pairs_fenwick(nums):
    n = len(nums)
    r = next_smaller_index(nums)
    fw = Fenwick(n)
    order = sorted(range(n), key=lambda i: -nums[i])   # chỉ số theo giá trị giảm dần
    answer = 0
    k = 0
    while k < n:
        # gom các chỉ số có cùng giá trị
        g = k
        while g < n and nums[order[g]] == nums[order[k]]:
            g += 1
        group = order[k:g]
        # truy vấn TRƯỚC khi đánh dấu nhóm (chỉ đếm phần tử lớn hơn hẳn)
        for i in group:
            # đoạn chỉ số (i, r[i]) ~ vị trí Fenwick (i+2 .. r[i]) vì Fenwick dùng 1-based
            answer += fw.prefix(r[i]) - fw.prefix(i + 1)
        for i in group:
            fw.add(i + 1)
        k = g
    return answer


print("r của [3,1,4,1,5]:", next_smaller_index([3, 1, 4, 1, 5]))
print(shadow_pairs_fenwick([3, 1, 4, 1, 5]))   # 3
print(shadow_pairs_fenwick([6, 7, 6, 6, 7]))   # 4
print(shadow_pairs_fenwick([1, 2, 3, 4]))      # 6

random.seed(5)
for _ in range(1000):
    nums = [random.randint(1, 5) for _ in range(random.randint(3, 10))]
    assert shadow_pairs_fenwick(nums) == shadow_pairs_cubic(nums)
print("Cách Fenwick khớp brute force ✔")

r của [3,1,4,1,5]: [1, 5, 3, 5, 5]
3
4
6
Cách Fenwick khớp brute force ✔


Cách 5 chạy `O(n log n)` — chậm hơn cách 4 và dài hơn nhiều. Nhưng nó dạy một kỹ thuật tổng quát: **"đếm số phần tử lớn hơn v trong một đoạn chỉ số" = xử lý offline theo giá trị giảm dần + Fenwick**. Kỹ thuật này dùng được cả khi bài không có cấu trúc stack đẹp như ở đây.

# 14. Những lỗi thường gặp

## Lỗi 1 — Để phần tử **bằng** chặn bóng

Điều kiện chặn là `nums[k] < nums[i]` (**nhỏ hơn hẳn**). Nếu pop cả phần tử `== x` (dùng `>=`), ví dụ 2 sai: số `6` ở `i = 0` bị số `6` ở `j = 2` loại mất.

## Lỗi 2 — Đếm cả phần tử bằng `x` là cặp

Cặp cần `nums[i] < nums[j]` nghiêm ngặt. Quên trừ nhóm `== x` ở đỉnh → đếm thừa.

## Lỗi 3 — Không gom nhóm → `O(n^2)` với mảng tăng dần / nhiều giá trị bằng nhau

## Lỗi 4 — Hiểu sai điều kiện 3 thành "không có phần tử nào ở giữa nhỏ hơn `nums[j]`"

Điều kiện so với `nums[i]`, không phải `nums[j]`.

In [15]:
def wrong_pop_equal(nums):
    stack, total, answer = [], 0, 0
    for x in nums:
        while stack and stack[-1][0] >= x:          # SAI: pop cả phần tử bằng x
            total -= stack.pop()[1]
        answer += total
        stack.append([x, 1])
        total += 1
    return answer

print("Sai (phần tử bằng chặn bóng):", wrong_pop_equal([6, 7, 6, 6, 7]))
print("Đúng                        :", sol.shadowPairs([6, 7, 6, 6, 7]))

Sai (phần tử bằng chặn bóng): 2
Đúng                        : 4


# 15. Mẹo nhận diện Monotonic Stack

Dấu hiệu:

- Điều kiện kiểu **"không có phần tử nào ở giữa nhỏ hơn / lớn hơn ..."**.
- Hỏi về **phần tử nhỏ hơn / lớn hơn gần nhất** bên trái/phải (Next Greater/Smaller Element).
- Một phần tử mới xuất hiện làm các phần tử cũ **"vô dụng vĩnh viễn"** → loại khỏi stack.

Template:

```python
stack = []
for j, x in enumerate(nums):
    while stack and <đỉnh stack bị x làm vô dụng>:
        stack.pop()          # (có thể ghi nhận thông tin ở đây)
    <dùng stack để trả lời câu hỏi cho j>
    stack.append(x)          # hoặc append(j), hoặc [x, count]
```

Mẹo phụ: khi có **nhiều giá trị bằng nhau**, hãy **gom nhóm** `[value, count]` và giữ **biến tổng** để đếm trong `O(1)`.

# 16. Biến thể — Shadow pair "ngược chiều"

Đổi đề: đếm cặp `i < j` với `nums[i] > nums[j]` và không có `k` ở giữa mà `nums[k] > nums[i]`.

Mẹo: **đổi dấu** mọi phần tử! `nums[i] > nums[j]` ⇔ `-nums[i] < -nums[j]`, và `nums[k] > nums[i]` ⇔ `-nums[k] < -nums[i]`. Vậy đáp án = `shadowPairs([-x for x in nums])`.

In [16]:
def reverse_shadow_bruteforce(nums):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n)
               if nums[i] > nums[j] and not any(nums[k] > nums[i] for k in range(i + 1, j)))

def reverse_shadow(nums):
    return sol.shadowPairs([-x for x in nums])

random.seed(16)
for _ in range(500):
    nums = [random.randint(1, 5) for _ in range(random.randint(3, 9))]
    assert reverse_shadow(nums) == reverse_shadow_bruteforce(nums)
print("Biến thể đổi dấu: khớp brute force ✔", reverse_shadow([4, 3, 2, 1]))

Biến thể đổi dấu: khớp brute force ✔ 6


# 17. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `nums = [2, 5, 1, 3, 4]`:

1. Vẽ "bóng" của từng phần tử (bị chặn ở đâu).
2. Liệt kê mọi shadow pair.
3. Lập bảng trace của cách 4 (stack gom nhóm) như mục 9.

## Bài 2 — Dự đoán trạng thái stack

Không chạy code, viết ra stack (dạng nhóm) sau mỗi bước với `nums = [4, 4, 2, 2, 3, 2]`. Sau đó kiểm tra bằng `trace_shadow`.

In [17]:
trace_shadow([4, 4, 2, 2, 3, 2])

j=0, x=4: pop - | cộng 0 | answer=0 | stack=[[4, 1]] | total=1
j=1, x=4: pop - | cộng 0 | answer=0 | stack=[[4, 2]] | total=2
j=2, x=2: pop [[4, 2]] | cộng 0 | answer=0 | stack=[[2, 1]] | total=1
j=3, x=2: pop - | cộng 0 | answer=0 | stack=[[2, 2]] | total=2
j=4, x=3: pop - | cộng 2 | answer=2 | stack=[[2, 2], [3, 1]] | total=3
j=5, x=2: pop [[3, 1]] | cộng 0 | answer=2 | stack=[[2, 3]] | total=3


2

## Bài 3 — Điền chỗ trống: cách 2 (dừng sớm)

In [18]:
def exercise_3_quadratic(nums):
    count = 0
    # TODO: với mỗi i, đi j sang phải:
    #   - nếu nums[j] < nums[i]: break
    #   - nếu nums[j] > nums[i]: count += 1
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")

# assert exercise_3_quadratic([3, 1, 4, 1, 5]) == 3

## Bài 4 — Tự viết cách 4

Viết lại stack gom nhóm mà không nhìn code. Kiểm tra với 3 ví dụ của đề.

In [19]:
def exercise_4_grouped_stack(nums):
    # TODO
    raise NotImplementedError

## Bài 5 — Next Greater Element

Với mỗi `i`, tìm chỉ số `j > i` **gần nhất** có `nums[j] > nums[i]` (không có thì `-1`). Đây là bài monotonic stack kinh điển.

Ví dụ: `[2, 1, 2, 4, 3]` → `[3, 2, 3, -1, -1]`.

In [20]:
def exercise_5_next_greater(nums):
    # TODO: stack chứa chỉ số; pop khi nums[đỉnh] < x
    raise NotImplementedError

## Bài 6 — Mini challenge: số shadow pair kết thúc tại mỗi `j`

Trả về list `ends` với `ends[j]` = số `i` sao cho `(i, j)` là shadow pair. Kiểm tra `sum(ends) == shadowPairs(nums)`.

In [21]:
def exercise_6_pairs_ending_at(nums):
    # TODO
    raise NotImplementedError

## Gợi ý theo mức độ

**Bài 4**
- Mức 1: stack chứa `[value, count]`, biến `total`.
- Mức 2: pop khi `stack[-1][0] > x` (chỉ `>`, không `>=`).
- Mức 3: nếu đỉnh `== x`: cộng `total - đỉnh.count` rồi tăng count; ngược lại cộng `total` và push `[x, 1]`. Luôn `total += 1`.

**Bài 5**
- Mức 1: duyệt `j` từ trái sang, stack chứa các `i` chưa tìm được "người lớn hơn".
- Mức 2: khi gặp `x`, mọi `i` ở đỉnh có `nums[i] < x` đã tìm được đáp án là `j`.
- Mức 3: `while stack and nums[stack[-1]] < x: res[stack.pop()] = j`.

**Bài 6**
- Mức 1: trong cách 4, giá trị "cộng" ở bước `j` chính là `ends[j]`.
- Mức 2: thay vì cộng vào `answer`, `append` vào list.

# 18. Đáp án tham khảo

**Bài 1:** `nums = [2, 5, 1, 3, 4]`

- Bóng của `2` (i=0): chặn tại `1` (k=2) → chỉ chứa `j=1` (5 > 2) → cặp `(0,1)`.
- Bóng của `5` (i=1): chặn ngay tại `1` (k=2) → không có cặp.
- Bóng của `1` (i=2): kéo tới hết → `(2,3)`, `(2,4)`.
- Bóng của `3` (i=3): `(3,4)`.

Tổng **4** cặp.

**Bài 2:** stack sau từng bước: `[[4,1]]` → `[[4,2]]` → `[[2,1]]` → `[[2,2]]` → `[[2,2],[3,1]]` → `[[2,3]]`. Đáp án = 2 (hai số 2 ở `i = 2, 3` cùng tạo cặp với số 3 ở `j = 4`).

In [22]:
def answer_3_quadratic(nums):
    count = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[j] < nums[i]:
                break
            if nums[j] > nums[i]:
                count += 1
    return count

assert answer_3_quadratic([3, 1, 4, 1, 5]) == 3
assert answer_3_quadratic([2, 5, 1, 3, 4]) == 4
print("Đáp án bài 3 PASS")

Đáp án bài 3 PASS


In [23]:
def answer_4_grouped_stack(nums):
    stack, total, answer = [], 0, 0
    for x in nums:
        while stack and stack[-1][0] > x:
            total -= stack.pop()[1]
        if stack and stack[-1][0] == x:
            answer += total - stack[-1][1]
            stack[-1][1] += 1
        else:
            answer += total
            stack.append([x, 1])
        total += 1
    return answer

for nums, exp in [([3, 1, 4, 1, 5], 3), ([6, 7, 6, 6, 7], 4), ([1, 2, 3, 4], 6)]:
    assert answer_4_grouped_stack(nums) == exp
print("Đáp án bài 4 PASS")

Đáp án bài 4 PASS


In [24]:
def answer_5_next_greater(nums):
    res = [-1] * len(nums)
    stack = []                         # chỉ số, giá trị không tăng
    for j, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:
            res[stack.pop()] = j
        stack.append(j)
    return res

assert answer_5_next_greater([2, 1, 2, 4, 3]) == [3, 2, 3, -1, -1]
print("Đáp án bài 5 PASS:", answer_5_next_greater([2, 1, 2, 4, 3]))

Đáp án bài 5 PASS: [3, 2, 3, -1, -1]


In [25]:
def answer_6_pairs_ending_at(nums):
    stack, total, ends = [], 0, []
    for x in nums:
        while stack and stack[-1][0] > x:
            total -= stack.pop()[1]
        if stack and stack[-1][0] == x:
            ends.append(total - stack[-1][1])
            stack[-1][1] += 1
        else:
            ends.append(total)
            stack.append([x, 1])
        total += 1
    return ends

e = answer_6_pairs_ending_at([6, 7, 6, 6, 7])
print("ends =", e, "| tổng =", sum(e))
assert sum(e) == sol.shadowPairs([6, 7, 6, 6, 7])
print("Đáp án bài 6 PASS")

ends = [0, 1, 0, 0, 3] | tổng = 4
Đáp án bài 6 PASS


# 19. Tổng kết

## Kiến thức Python

```python
stack.append(x), stack.pop(), stack[-1], if stack:
any(cond for ...)            # có ít nhất một?
sorted(range(n), key=lambda i: -nums[i])
pos & (-pos)                 # bit thấp nhất (Fenwick)
```

## Kỹ thuật thuật toán

1. **Rút gọn điều kiện** (điều kiện 3 chỉ còn "không ai ở giữa nhỏ hơn `nums[i]`").
2. **Đổi góc nhìn**: cố định `j`, theo dõi các `i` còn sống.
3. **Monotonic stack** không giảm: pop các ứng viên bị chặn.
4. **Gom nhóm + biến tổng**: đếm "số phần tử `< x`" trong `O(1)`.
5. (Mở rộng) Next Smaller Element + Fenwick, xử lý offline theo giá trị.

## Công thức nhớ nhanh

> Pop mọi nhóm `> x`; cộng `total` trừ nhóm `== x` ở đỉnh; gộp/push `x`; `total += 1`.

## Độ phức tạp

```text
Time:  O(n)    (khấu hao)
Space: O(n)
```

## Checklist trước khi nộp

- [ ] Pop với `>` (không phải `>=`)?
- [ ] Trừ nhóm bằng `x` khi đếm?
- [ ] Gom nhóm để tránh `O(n^2)`?
- [ ] Không đếm từng cặp (đáp án có thể ~5·10^9)?

# 20. Phiên bản cuối cùng nên nhớ

```python
class Solution:
    def shadowPairs(self, nums: list[int]) -> int:
        stack = []      # [value, count]
        total = 0
        answer = 0
        for x in nums:
            while stack and stack[-1][0] > x:
                total -= stack.pop()[1]
            if stack and stack[-1][0] == x:
                answer += total - stack[-1][1]
                stack[-1][1] += 1
            else:
                answer += total
                stack.append([x, 1])
            total += 1
        return answer
```

Hiểu được vì sao stack **luôn không giảm** và vì sao phần tử **bằng nhau nằm ở đỉnh**, bạn sẽ tự viết được lời giải này cho nhiều bài monotonic stack khác.